In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from google.cloud import bigquery
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# TEST BIGQUERY CONNECTION
# ============================================================

client = bigquery.Client(project="project-a13303d6-d497-41fa-ab8")
print("✅ Connected to BigQuery successfully!")
print(f"Project: {client.project}")

✅ Connected to BigQuery successfully!
Project: project-a13303d6-d497-41fa-ab8


In [2]:
# ============================================================
# PULL MULTI-ASSET PRICE DATA
# ============================================================

# Define assets
assets = {
    "Oil":     "BZ=F",
    "Gold":    "GC=F",
    "SP500":   "^GSPC",
    "Bitcoin": "BTC-USD"
}

# Pull 2 years of daily data
all_data = []

for name, ticker in assets.items():
    print(f"Pulling {name} ({ticker})...")
    df = yf.download(ticker, start="2023-01-01", auto_adjust=True, progress=False)
    df = df[["Close", "Open", "High", "Low", "Volume"]].copy()
    df.columns = ["close", "open", "high", "low", "volume"]
    df["asset"] = name
    df["ticker"] = ticker
    df.index.name = "date"
    df = df.reset_index()
    all_data.append(df)
    print(f"✅ {name}: {len(df)} rows")

# Combine all assets
prices = pd.concat(all_data, ignore_index=True)
prices["date"] = pd.to_datetime(prices["date"]).dt.date

# Calculate daily return
prices = prices.sort_values(["asset", "date"])
prices["daily_return"] = prices.groupby("asset")["close"].pct_change() * 100

print(f"\n✅ Total rows: {len(prices)}")
print(prices.head(10))

Pulling Oil (BZ=F)...
✅ Oil: 802 rows
Pulling Gold (GC=F)...
✅ Gold: 801 rows
Pulling SP500 (^GSPC)...
✅ SP500: 799 rows
Pulling Bitcoin (BTC-USD)...
✅ Bitcoin: 1166 rows

✅ Total rows: 3568
            date         close          open          high           low  \
2402  2023-01-01  16625.080078  16547.914062  16630.439453  16521.234375   
2403  2023-01-02  16688.470703  16625.509766  16759.343750  16572.228516   
2404  2023-01-03  16679.857422  16688.847656  16760.447266  16622.371094   
2405  2023-01-04  16863.238281  16680.205078  16964.585938  16667.763672   
2406  2023-01-05  16836.736328  16863.472656  16884.021484  16790.283203   
2407  2023-01-06  16951.968750  16836.472656  16991.994141  16716.421875   
2408  2023-01-07  16955.078125  16952.117188  16975.017578  16914.191406   
2409  2023-01-08  17091.144531  16954.146484  17091.144531  16924.050781   
2410  2023-01-09  17196.554688  17093.992188  17389.957031  17093.992188   
2411  2023-01-10  17446.292969  17192.949219  174

In [3]:
# ============================================================
# WRITE PRICES TO BIGQUERY
# ============================================================

table_id = "project-a13303d6-d497-41fa-ab8.market_intelligence.prices"

# Define schema
job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("date", "DATE"),
        bigquery.SchemaField("asset", "STRING"),
        bigquery.SchemaField("ticker", "STRING"),
        bigquery.SchemaField("open", "FLOAT"),
        bigquery.SchemaField("high", "FLOAT"),
        bigquery.SchemaField("low", "FLOAT"),
        bigquery.SchemaField("close", "FLOAT"),
        bigquery.SchemaField("volume", "FLOAT"),
        bigquery.SchemaField("daily_return", "FLOAT"),
    ],
    write_disposition="WRITE_TRUNCATE"  # overwrite table if exists
)

# Upload to BigQuery
job = client.load_table_from_dataframe(prices, table_id, job_config=job_config)
job.result()  # Wait for job to complete

# Verify
table = client.get_table(table_id)
print(f"✅ Prices table created!")
print(f"Rows in BigQuery: {table.num_rows}")
print(f"Columns: {[field.name for field in table.schema]}")

✅ Prices table created!
Rows in BigQuery: 3568
Columns: ['date', 'close', 'open', 'high', 'low', 'volume', 'asset', 'ticker', 'daily_return']


In [4]:
# ============================================================
# QUERY BIGQUERY WITH SQL
# ============================================================

query = """
    SELECT 
        asset,
        COUNT(*) as total_rows,
        MIN(date) as earliest_date,
        MAX(date) as latest_date,
        ROUND(AVG(close), 2) as avg_close,
        ROUND(AVG(daily_return), 4) as avg_daily_return
    FROM `project-a13303d6-d497-41fa-ab8.market_intelligence.prices`
    GROUP BY asset
    ORDER BY asset
"""

results = client.query(query).to_dataframe()
print("=== BIGQUERY QUERY RESULTS ===")
print(results.to_string(index=False))

=== BIGQUERY QUERY RESULTS ===
  asset  total_rows earliest_date latest_date  avg_close  avg_daily_return
Bitcoin        1166    2023-01-01  2026-03-11   66264.02            0.1553
   Gold         801    2023-01-03  2026-03-11    2731.30            0.1367
    Oil         802    2023-01-03  2026-03-11      76.33            0.0297
  SP500         799    2023-01-03  2026-03-11    5403.14            0.0758


In [5]:
# ============================================================
# ADD VOLATILITY AND UPDATE BIGQUERY
# ============================================================

# Calculate 30 day rolling volatility per asset
prices = prices.sort_values(["asset", "date"])
prices["volatility_30"] = prices.groupby("asset")["daily_return"].transform(
    lambda x: x.rolling(window=30).std()
)

# Update BigQuery table
job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("date", "DATE"),
        bigquery.SchemaField("asset", "STRING"),
        bigquery.SchemaField("ticker", "STRING"),
        bigquery.SchemaField("open", "FLOAT"),
        bigquery.SchemaField("high", "FLOAT"),
        bigquery.SchemaField("low", "FLOAT"),
        bigquery.SchemaField("close", "FLOAT"),
        bigquery.SchemaField("volume", "FLOAT"),
        bigquery.SchemaField("daily_return", "FLOAT"),
        bigquery.SchemaField("volatility_30", "FLOAT"),
    ],
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(prices, table_id, job_config=job_config)
job.result()

print("✅ Volatility added and BigQuery updated!")

# Verify with a query
query = """
    SELECT 
        asset,
        ROUND(AVG(volatility_30), 4) as avg_volatility,
        ROUND(MAX(volatility_30), 4) as max_volatility
    FROM `project-a13303d6-d497-41fa-ab8.market_intelligence.prices`
    WHERE volatility_30 IS NOT NULL
    GROUP BY asset
    ORDER BY avg_volatility DESC
"""

vol_results = client.query(query).to_dataframe()
print("\n=== VOLATILITY BY ASSET ===")
print(vol_results.to_string(index=False))

✅ Volatility added and BigQuery updated!

=== VOLATILITY BY ASSET ===
  asset  avg_volatility  max_volatility
Bitcoin          2.3955          4.4290
    Oil          1.8039          3.8639
   Gold          1.0667          3.0732
  SP500          0.8556          2.6932
